# 00 — Environment Setup
## CIS Windows 11 Benchmark RAG Workshop

This notebook prepares the local project environment for the RAG architectures we will implement one by one.

### Learning objectives
- Confirm Python and Jupyter are running in the intended environment.
- Create and verify the workshop directory structure.
- Install the project dependencies when you are ready.
- Configure local Ollama access (optional) and check available models.
- Confirm the embedding and vector-store packages import correctly.

**Important:** Run this notebook from the project environment you intend to use for the workshop. On Windows, activate your virtual environment before starting Jupyter. Do not put API keys or passwords directly in notebook cells.

## 1. Project structure

The notebook expects this simplified structure:

```text
cis-windows11-rag/
├── data/
│   ├── benchmark/
│   ├── machine_evidence/
│   └── evaluation/
├── notebooks/
│   └── 00_environment_setup.ipynb
├── vector_store/
├── evaluation_results/
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md
```

Place the CIS Windows 11 Benchmark PDF in `data/benchmark/`. Use only documents you are authorized to access and process.

In [ ]:
from pathlib import Path
import os
import sys
import platform

# Resolve the project root from this notebook's location.
# This works when the notebook is stored in the expected /notebooks directory.
NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "notebooks").exists():
    # Also supports launching Jupyter from the project root.
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    # A clear fallback for interactive environments; review this path if needed.
    PROJECT_ROOT = NOTEBOOK_DIR

print(f"Python version : {sys.version.split()[0]}")
print(f"Operating system: {platform.platform()}")
print(f"Python executable: {sys.executable}")
print(f"Current folder : {NOTEBOOK_DIR}")
print(f"Project root   : {PROJECT_ROOT}")

## 2. Create the required folders

This cell is safe to run repeatedly. It creates missing directories without deleting existing files.

In [ ]:
# Keep all workshop artifacts in the agreed project layout.
DIRECTORIES = [
    "data/benchmark",
    "data/machine_evidence",
    "data/evaluation",
    "notebooks",
    "vector_store",
    "evaluation_results",
]

for relative_path in DIRECTORIES:
    folder = PROJECT_ROOT / relative_path
    folder.mkdir(parents=True, exist_ok=True)
    print(f"✓ {folder.relative_to(PROJECT_ROOT)}")

print("\\nDirectory setup complete.")

## 3. Install dependencies

The project `requirements.txt` includes the core LangChain integrations, local embeddings, FAISS/Chroma, keyword retrieval, RAGAS evaluation, and graph support for later lessons. Some optional multimodal tools may be added when we reach that notebook.

The installation cell is **disabled by default** to avoid unexpectedly changing your Python environment. Set `RUN_INSTALL = True` only when you are ready. It installs into the Python environment running this notebook.

In [ ]:
# Change to True to install or update the packages listed in requirements.txt.
RUN_INSTALL = False

requirements_file = PROJECT_ROOT / "requirements.txt"

if not requirements_file.exists():
    raise FileNotFoundError(
        f"Could not find {requirements_file}. "
        "Make sure this notebook is inside the project/notebooks folder."
    )

print(f"Requirements file: {requirements_file}")
if RUN_INSTALL:
    import subprocess
    # Use this notebook's Python executable so packages go into the active kernel environment.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r", str(requirements_file)
    ])
    print("Dependency installation finished. Restart the notebook kernel if imports fail.")
else:
    print("Installation skipped. Set RUN_INSTALL = True and rerun this cell when ready.")
    print(f'Command-line alternative: "{sys.executable}" -m pip install -r "{requirements_file}"')

## 4. Check core package availability

Run this after installing the dependencies. Missing packages will be reported clearly.

In [ ]:
# Each package is checked independently so one missing dependency does not hide the others.
from importlib.util import find_spec

PACKAGE_CHECKS = {
    "langchain": "langchain",
    "langchain_core": "langchain_core",
    "langchain_community": "langchain_community",
    "langchain_ollama": "langchain_ollama",
    "langchain_huggingface": "langchain_huggingface",
    "langchain_text_splitters": "langchain_text_splitters",
    "pypdf": "pypdf",
    "sentence_transformers": "sentence_transformers",
    "faiss": "faiss",
    "chromadb": "chromadb",
    "rank_bm25": "rank_bm25",
    "ragas": "ragas",
    "datasets": "datasets",
    "networkx": "networkx",
    "dotenv": "dotenv",
}

missing_packages = []
for display_name, import_name in PACKAGE_CHECKS.items():
    available = find_spec(import_name) is not None
    print(f"{'✓' if available else '✗'} {display_name}")
    if not available:
        missing_packages.append(display_name)

if missing_packages:
    print("\\nMissing packages:", ", ".join(missing_packages))
    print("Install dependencies using the previous cell, then restart the kernel.")
else:
    print("\\nAll checked packages are available.")

## 5. Configure local Ollama (optional)

This workshop can use a local Ollama model for chat generation. We will keep model settings in environment variables so the notebook code is easy to adjust.

1. Install and start Ollama separately if it is not already installed.
2. Pull a model you are allowed to use, for example `gemma4:e2b`.
3. Confirm it appears in `ollama list`.
4. Set the model name below to match the name on your machine.

The chat model and embedding model are separate components. The default embedding model below is a local Sentence Transformers model; its first use may download model files unless they are already cached.

In [ ]:
# Read configuration from environment variables when available.
# These defaults are examples and can be changed for your machine.
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(f"Ollama URL     : {OLLAMA_BASE_URL}")
print(f"Ollama model   : {OLLAMA_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")

In [ ]:
# Check whether the Ollama command-line application is installed and can list models.
# This check does not start Ollama or download a model.
import shutil
import subprocess

ollama_executable = shutil.which("ollama")

if ollama_executable is None:
    print("Ollama CLI was not found on PATH. Skip this check if you plan to use another model provider.")
else:
    try:
        result = subprocess.run(
            [ollama_executable, "list"],
            capture_output=True,
            text=True,
            timeout=15,
            check=False,
        )
        print(result.stdout.strip() or result.stderr.strip())
        if result.returncode != 0:
            print("Ollama returned a non-zero exit code. Confirm Ollama is installed and running.")
    except subprocess.TimeoutExpired:
        print("The Ollama command timed out. Check the Ollama installation and service.")

## 6. Smoke-test Ollama chat (optional)

This cell sends one short test prompt to the configured local model. Run it only if Ollama is installed, running, and the selected model is available. It may take some time on the first run.

In [ ]:
RUN_OLLAMA_TEST = False  # Set to True only after confirming the model is available.

if RUN_OLLAMA_TEST:
    try:
        from langchain_ollama import ChatOllama

        chat_model = ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
            temperature=0,
        )
        response = chat_model.invoke(
            "Reply with exactly this text: CIS RAG setup successful."
        )
        print(response.content)
    except Exception as exc:
        print("Ollama smoke test failed.")
        print("Check that Ollama is running, the model name is correct, and the package is installed.")
        print(f"Technical detail: {type(exc).__name__}: {exc}")
else:
    print("Ollama smoke test skipped. Set RUN_OLLAMA_TEST = True to run it.")

## 7. Check the embedding model (optional)

This test loads the configured local embedding model and creates a vector for a short sentence. The first run can download model files. If you need a fully offline setup, download/cache the model in advance.

In [ ]:
RUN_EMBEDDING_TEST = False  # Set to True when the dependencies and model download are ready.

if RUN_EMBEDDING_TEST:
    try:
        from langchain_huggingface import HuggingFaceEmbeddings

        embedding_model = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL
        )
        vector = embedding_model.embed_query(
            "CIS recommends a secure Windows configuration."
        )
        print(f"Embedding dimensions: {len(vector)}")
        print(f"First five values  : {vector[:5]}")
    except Exception as exc:
        print("Embedding test failed. Check package installation and model availability.")
        print(f"Technical detail: {type(exc).__name__}: {exc}")
else:
    print("Embedding test skipped. Set RUN_EMBEDDING_TEST = True to run it.")

## 8. Locate the CIS Benchmark PDF

This cell lists PDF files in `data/benchmark/`. It does not assume a particular filename.

In [ ]:
benchmark_dir = PROJECT_ROOT / "data" / "benchmark"
pdf_files = sorted(benchmark_dir.glob("*.pdf"))

if pdf_files:
    print("PDF files found:")
    for pdf_path in pdf_files:
        print(f" - {pdf_path.name}")
else:
    print(f"No PDF found in {benchmark_dir}")
    print("Copy your authorized CIS Windows 11 Benchmark PDF into this folder before starting Naive RAG.")

## 9. Setup checklist

Use this checklist before moving to `01_naive_rag.ipynb`.

- [ ] Notebook is running in the intended Python environment.
- [ ] Project directories have been created.
- [ ] Dependencies are installed and the kernel has been restarted if necessary.
- [ ] The selected chat model is available, or an alternative provider is configured.
- [ ] The embedding model can load.
- [ ] The CIS Benchmark PDF is present in `data/benchmark/`.
- [ ] No API keys or sensitive information have been committed to source control.

### What comes next?

In `01_naive_rag.ipynb`, we will load the PDF, inspect its pages, split it into chunks, create embeddings, persist a vector index, retrieve relevant passages, and generate a source-grounded answer. We will keep each stage in a separate, commented cell so it is easy to inspect and troubleshoot.